# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

if( !require("yaml")) install.packages("yaml")
require("yaml")

if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

if( !require("primes")) install.packages("primes")
require("primes")

#### Parametros

In [ ]:
PARAM <- list()

PARAM$experimento <- 9100
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

PARAM$ntfy_topic <- "exp-colab-jr-tomi-ale" # nuevo param agregado - topico a notificar novedades

In [ ]:
# Semillas
PARAM$semilla_primigenia <- 100151 # 100151 (ale) / 999043 (tomi)
PARAM$qsemillas <- 7 # cantidad de semillas a generar

# a partir de la `semilla_primigenia` se genera, de forma reproducible, la lista de `PARAM$qsemillas` semillas que se van a usar para correr cada experimento.

# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas)

PARAM$semillas

#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 9.3.1   Preprocesamiento del dataset

#### 9.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 9.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

In [ ]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [ ]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [ ]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [ ]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [ ]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [ ]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

#### 9.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, ajustando por algunos indices financieros

In [ ]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [ ]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [ ]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

In [ ]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [ ]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [ ]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [ ]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [ ]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [ ]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [ ]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [ ]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

In [ ]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "deflacion"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


In [ ]:
colnames(dataset)

In [ ]:
# se intenta corregir el data drifting utilizando algunos indices financieros

#### 9.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

#### 9.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 9.3.1.5  FEhist Feature Engineering historico - **Experimento elegido**

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

> Se adapta esta sección con los siguientes cambios:
> - Se crea una matriz que contiene parametros de los experimentos a ejecutar
> - Se crea una función que contiene cada experimento a ejecutar

In [ ]:
PARAM$experiments <- list(
  # en este experimento, no se agrega ninguna columna nueva --> ya de por si, sabemos que va a ser el peor experimento. Se decide no ejecutarlo
  list(
    name          = "base", # en este experimento, no se agrega ninguna columna nueva
    enabled       = FALSE,   # indica si el experimento va a ser ejecutado o no
    lag_orders    = NULL,    # no aplica
    roll_windows  = NULL,   # no aplica
    delta         = FALSE, #no aplica
    ratio         = FALSE,  # no aplica
    selective     = FALSE,  # no aplica
    var_threshold = NULL    # no aplica
  ),

  # ESTE ES EL WORKFLOW BASELINE (el que viene en el script por defecto)
  list(
    name        = "lags_deltas", # en este experimento, se agregan N columnas de lag y N columnas de delta por cada atributo
    enabled     = TRUE, # indica si el experimento va a ser ejecutado o no
    lag_orders  = c(1, 2), # indica cuantas columnas de lag se crean por cada atributo
    roll_windows= NULL,   # no aplica
    delta       = TRUE, # indica que se agregan los deltas
    ratio       = FALSE, # no aplica
    selective   = FALSE, # no aplica
    var_threshold = NULL # no aplica
  ),

  list(
    name        = "rolling", # en este experimento, se agregan una columna de media y una de std por cada atributo, mirando N meses hacia atras
    enabled     = TRUE, # indica si el experimento va a ser ejecutado o no
    lag_orders  = NULL, # no aplica
    roll_windows= c(3, 6), # cuantos meses hacia atras mira
    delta       = FALSE, # no aplica
    ratio       = TRUE, # no aplica
    selective   = FALSE, # no aplica
    var_threshold = NULL # no aplica
  ),

  list(
    name        = "lags_ratio",  # en este experimento, se agregan N columnas de lag y N columnas de ratio por cada atributo
    enabled     = TRUE, # indica si el experimento va a ser ejecutado o no
    lag_orders  = c(1, 2),  # indica cuantas columnas de lag se crean por cada atributo
    roll_windows= NULL, # no aplica
    delta       = FALSE, # no aplica
    ratio       = TRUE, # indica que se agregan los ratios
    selective   = FALSE, # no aplica
    var_threshold = NULL # no aplica
  ),

  list(
    name        = "selective", #
    enabled     = TRUE, # indica si el experimento va a ser ejecutado o no en el main loop
    lag_orders  = NULL, # no aplica
    roll_windows= NULL, # no aplica
    delta       = FALSE, # no aplica
    ratio       = FALSE, # no aplica
    selective   = TRUE, # indica que se aplique este experimento
    var_threshold = 1e-4
  )
)

In [ ]:
# Feature Engineering Historico

FE_Hist <- function(data, exp_cfg) {

  # todo es lagueable, menos la primary key y la clase
  cols_lagueables <- copy(setdiff(
    colnames(data),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
  ))

  # ── selective: filter low-variance columns ──────────────────
  if (isTRUE(exp_cfg$selective)) {
    col_vars <- data[, lapply(.SD, var, na.rm = TRUE), .SDcols = cols_lagueables]
    cols_lagueables <- names(col_vars)[as.numeric(col_vars[1,]) > exp_cfg$var_threshold]
    cat(sprintf("  [selective] %d cols kept after variance filter\n", length(cols_lagueables)))
  }

  # ── lags ────────────────────────────────────────────────────
  # https://rdrr.io/cran/data.table/man/shift.html
  for (lag_n in exp_cfg$lag_orders) {
    data[,
      paste0(cols_lagueables, paste0("_lag", lag_n)) := shift(.SD, lag_n, NA, "lag"),
      by = numero_de_cliente,
      .SDcols = cols_lagueables
    ]
  }

  # ── deltas ──────────────────────────────────────────────────
   if (isTRUE(exp_cfg$delta)) {
    for (vcol in cols_lagueables) {
      for (lag_n in exp_cfg$lag_orders) {
        data[, paste0(vcol, "_delta", lag_n) :=
          get(vcol) - get(paste0(vcol, "_lag", lag_n))]
      }
    }
   }

  # ── ratios ──────────────────────────────────────────────────
  if (isTRUE(exp_cfg$ratio)) {
    EPSILON <- 1e-8
    for (vcol in cols_lagueables) {
      for (lag_n in exp_cfg$lag_orders) {
        data[, paste0(vcol, "_ratio", lag_n) :=
          get(vcol) / (get(paste0(vcol, "_lag", lag_n)) + EPSILON)]
      }
    }
  }

  # ── rolling mean + sd ────────────────────────────────────────
  for (w in exp_cfg$roll_windows) {

    data[,
      paste0(cols_lagueables, paste0("_rollmean", w)) :=
        lapply(.SD, function(x) frollmean(x, n = w, align = "right", na.rm = TRUE)),
      by = numero_de_cliente,
      .SDcols = cols_lagueables
    ]

    data[,
      paste0(cols_lagueables, paste0("_rollsd", w)) :=
        lapply(.SD, function(x) {
          sqrt(pmax(
            frollmean(x^2, n = w, align = "right", na.rm = TRUE) -
            frollmean(x,   n = w, align = "right", na.rm = TRUE)^2,
            0
          ))
        }),
      by = numero_de_cliente,
      .SDcols = cols_lagueables
    ]
  }

  cat(sprintf("  [FE_Hist] done — total cols: %d\n", ncol(data)))
  return(data)
}

Verificacion de los campos recien creados

#### 9.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  ni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 9.3.2 Modelado

#### 9.3.2.1 Training Strategy

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 201901, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 201901, 202105 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
generate_model <- function(data) {
    # seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
    data[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

    # los campos en los que se entrena
    campos_buenos <- copy( setdiff(
        colnames(data), c("clase_ternaria","clase01","azar"))
    )

    # preparo para que se puede hacer undersampling de los CONTINUA
    #  solamente por un tema de VELOCIDAD
    set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
    data[, azar:=runif(nrow(data))]

    # undersampling de los CONTINUA
    data[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
        (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
         azar < PARAM$trainingstrategy$training_pct ) ]

    dtrain <- lgb.Dataset(
      data= data.matrix(data[fold_train == TRUE, campos_buenos, with = FALSE]),
      label= data[fold_train == TRUE, clase01],
      free_raw_data= TRUE
    )

    # datos de validation
    dvalidate <- lgb.Dataset(
      data= data.matrix(data[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
      label= data[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
      free_raw_data= TRUE
    )

    return (list(
        campos_buenos = campos_buenos,
        dtrain = dtrain,
        dvalidate = dvalidate))
}

####  9.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x, dtrain, dvalidate) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  # message(format(Sys.time(), "%a %b %d %X %Y  "),
  #   toString(x),
  #   " niter ", modelo_train$best_iter,
  #   " AUC ", AUC
  # )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 384, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048),
  feature_fraction= c(0.5, 0.8)
)

##### Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 65 minutos
<br> una Analista Jr  debe ser capaz de tolerar estoicamente esta tortura
<br> (y masticar chicle al mismo tiempo)

In [ ]:
calc_best_hyperparams <- function(experiment_name, dtrain, dvalidate) {
  # registro a registro calculo la AUC
  auc_table <- copy(tb_nueva)
  auc_table[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD, dtrain, dvalidate ),
    by=1:nrow(auc_table) ]

  # la optimizacion de hiperparámetros de tipo Grid Search ha corrido, extraigo los mejores hiperparametros

  fwrite(auc_table,
    file= sprintf("tb_grid_search_%s_01.txt", experiment_name),
    sep="\t",
    append= TRUE
  )

  setorder( auc_table, -AUC)  # ordeno DESCENDENTE por AUC
  best_hyperparams <- as.list(auc_table[1]) # en la posicion 1 estan los mejores
  best_hyperparams$AUC <- NULL
  return (list(
      best_auc = auc_table[1, AUC],
      best_hyperparams = best_hyperparams
  ))
}

### 9.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

In [ ]:
train_final_model <- function(experiment_name, data, campos_buenos, best_hyperparams) {

    data[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

    # creo el dfinal_train en formato  LightGBM
    dfinal_train <- lgb.Dataset(
      data= data.matrix(data[fold_final_train == TRUE, campos_buenos, with= FALSE]),
      label= data[fold_final_train == TRUE, clase01],
      free_raw_data= TRUE
    )

    # uno los parametros fijos y los mejores encontrados de los variables
    fijos <- copy(PARAM$lgbm$param_fijos)

    # quito lo que optimice en la Bayesian Optimization
    fijos$num_iterations <- NULL
    fijos$early_stopping_rounds <- NULL

    # agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
    param_final <- c(fijos, best_hyperparams)

    # y genero el modelo final, sin hacer ningun tipo de undersampling de la clase mayoritaria
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_final,
      verbose= -100
    )

    # grabo a disco el modelo en un formato para seres humanos ... ponele ...
    model_file_name <- sprintf("modelo_%s.txt", experiment_name)
    lgb.save(final_model, model_file_name)

    # ahora imprimo la importancia de variables
    tb_importancia <- as.data.table(lgb.importance(final_model))
    archivo_importancia <- sprintf("impo_%s.txt", experiment_name)

    fwrite( tb_importancia,
      file= archivo_importancia,
      sep= "\t"
    )

    return (final_model)
}

#### Scoring

Aplico el modelo final a los datos del futuro

In [ ]:
PARAM$trainingstrategy$future <- c(202109)

In [ ]:
predict_and_save <- function(experiment_name, data, final_model, campos_buenos) {
    dfuture <- data[ foto_mes %in% PARAM$trainingstrategy$future ]

    # aplico final_model   a dfuture
    prediccion <- predict(
      final_model,
      data.matrix(dfuture[, campos_buenos, with= FALSE])
    )

    # tabla prediccion
    tb_prediccion <- dfuture[, list(numero_de_cliente)]
    tb_prediccion[, prob := prediccion]

    # grabo las probabilidad del modelo
    #  me va a ser util para hacer Ensembles de modelos
    fwrite(tb_prediccion,
      file= sprintf("prediccion_%s.txt", experiment_name),
      sep= "\t"
    )
    return (tb_prediccion)
}

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 100)

In [ ]:
kaggle_submit <- function(experiment_name, tb_prediccion, params) {
    # ordeno por probabilidad descendente
    setorder(tb_prediccion, -prob)

    dir.create("kaggle", showWarnings= FALSE)

    for (envios in PARAM$kaggle$cortes) {

      tb_prediccion[, Predicted := 0L] # seteo inicial a 0
      tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

      archivo_kaggle <- sprintf(
          "./kaggle/KA%d_%s_%d.csv",
          PARAM$experimento,
          experiment_name,
          envios
      )

      # grabo el archivo
      fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
        file= archivo_kaggle,
        sep= ","
      )

      # subida a Kaggle, armo la linea de comando
      kaggle <- file.path(Sys.getenv("HOME"), ".venv", "bin", "kaggle")
      comando <- paste(shQuote(kaggle), "competitions submit")
      competencia <- paste("-c", PARAM$kaggle$competencia)
      arch <- paste( "-f", archivo_kaggle)

      mensaje <- sprintf(
          "-m 'experimento=%s envios=%d semilla=%d \n\nParametros:\n %s'",
          experiment_name,
          envios,
          PARAM$semilla_primigenia,
          hyperparams_to_string(params)
      )

      linea <- paste( comando, competencia, arch, mensaje)

      salida <- system(linea, intern=TRUE) # el submit a Kaggle
      cat(salida, "\n")
      flush.console()
      Sys.sleep(30)
    }
}

In [ ]:
save_experiment_results <- function(experiment_name, exp_start, best_params) {
    # grabo los parametros
    params_file_name <- sprintf("PARAM_%s.yml", experiment_name)
    write_yaml(PARAM, file=params_file_name)

    # grabo el resultado del experimento
    results_table <- data.table(
      experiment = experiment_name,
      best_auc = best_params$best_auc,
      num_leaves = best_params$best_hyperparams$num_leaves,
      min_data_in_leaf = best_params$best_hyperparams$min_data_in_leaf,
      feature_fraction = best_params$best_hyperparams$feature_fraction,
      num_iterations = best_params$best_hyperparams$num_iterations
    )
    summary_file_name <- sprintf("results_summary_%s.txt", PARAM$experimento)
    fwrite(results_table, summary_file_name, sep = "\t", append=TRUE)


    # notifico finalizacion de la ejecucion
    exp_end <- Sys.time()
    tiempo_ejecucion <- as.numeric(difftime(exp_end, exp_start, units = "mins"))

    ntfy_msg <- sprintf(
        "Experiment %s using seed %d ended - took %.2f minutes",
        experiment_name,
        PARAM$semilla_primigenia,
        tiempo_ejecucion)
    ntfy_send(ntfy_msg, topic = PARAM$ntfy_topic)
}

### Ejecución de todos los experimentos (9.3.1.5 FEhist Feature Engineering historico)

In [ ]:
hyperparams_to_string <- function(params) {
    msg <- sprintf(
        "num_leaves=%d, min_data_in_leaf=%d, feature_fraction=%.2f, num_iterations=%d - AUC = %.10f",
        params$best_hyperparams$num_leaves,
        params$best_hyperparams$min_data_in_leaf,
        params$best_hyperparams$feature_fraction,
        params$best_hyperparams$num_iterations,
        params$best_auc
    )
    return (msg)
}

In [ ]:
# por cada uno de los experimentos definidos en esa seccion
for (exp in PARAM$experiments) {

    if (!isTRUE(exp$enabled)) {
        cat(sprintf("\n[SKIP] experiment '%s' is disabled\n", exp$name))
        flush.console()
        next
    }

    cat(sprintf("\n========= Starting experiment %s =========\n", exp$name))
    flush.console()

    # por cada una de las semillas generadas en PARAM$semillas, se corre el experimento completo
    for (semilla in PARAM$semillas) {

        # nombre del experimento + semilla
        exp_name_semilla <- paste0(exp$name, "_s", semilla)

        cat(sprintf("\n----- experimento '%s'  semilla %d -----\n", exp$name, semilla))
        flush.console()

        # piso la semilla primigenia y el seed de LightGBM con la semilla de esta corrida
        PARAM$semilla_primigenia <- semilla
        PARAM$lgbm$param_fijos$seed <- semilla

        best <- NULL
        exp_dataset <- NULL
        exp_start <- Sys.time()

        # se simula la ejecucion completa del notebook
        tryCatch({
            # se ejecuta el experimento con el dataset baseline (esto es, con todos los experimentos externos tal cual como vinieron) + el experimento en cuestion
            exp_dataset <- FE_Hist(copy(dataset), exp)

            # modelado: particion train (hasta mayo 202105) / test (hasta julio 202107)
            model <- generate_model(exp_dataset)
            cat("model generated\n")
            flush.console()

            # Hyperparameter Tuning: grid search para encontrar los mejores hiperparametros con el model
            exp_best_hyperparams <- calc_best_hyperparams(exp_name_semilla, model$dtrain, model$dvalidate)
            cat("best hyperparams found:\n")
            cat(hyperparams_to_string(exp_best_hyperparams), '\n')
            flush.console()

            # Final training: entreno en todos los meses hasta julio (202107)
            final_model <- train_final_model(
                exp_name_semilla,
                exp_dataset,
                model$campos_buenos,
                exp_best_hyperparams$best_hyperparams
            )
            cat("\nfinal model trained successfully\n")
            flush.console()

            # Aplico el modelo final a los datos del futuro: ultimo mes que tenemos --> septiembre (202109)
            prediction <- predict_and_save(exp_name_semilla, exp_dataset, final_model, model$campos_buenos)
            cat("predictions saved\n")
            flush.console()

            # subo a Kaggle
            cat("Submitting results:\n")
            kaggle_submit(exp_name_semilla, prediction, exp_best_hyperparams)

            # guardamos resultados en el bucket de la instancia
            save_experiment_results(exp_name_semilla, exp_start, exp_best_hyperparams)
            cat("experiment submitted and saved successfully\n")
            flush.console()

        }, error = function(e) {
          cat(sprintf("\n[ERROR] experiment '%s' (semilla %d) failed: %s\n", exp$name, semilla, conditionMessage(e)))
          flush.console()
        })

        # cleanup memory
        rm(exp_dataset, model, final_model, prediction)
        gc(full = TRUE, verbose = FALSE)
    }
}


Una vez subido a Kaggle, reunimos las **ganancias** de **cada una** de las N **semillas** de **cada experimento** (vector de ganancias por experimento).
Luego, se realizará el **test de Wilcoxon** comparando los vectores de ganancias para determinar cuál fue el mejor experimento.

```R
  res <- wilcox.test(
    vector_ganancias_experimento_A,
    vector_ganancias_experimento_B,
    paired = TRUE # cada registro (el primero de vector_ganancias_experimento_A y el primero de vector_ganancias_experimento_B) se corrio con la misma semilla
  )
```